# UFC Fight Predictor — Step 2: Leakage-free features
**Rule for this whole notebook:** every feature for a fight may only use information from that fighter's *earlier* fights. Nothing from the fight itself, nothing from later.

Input: `data/ufc_clean.csv` (made by notebook 1). Output: `data/ufc_features.csv`, one row per fight with features for fighter A, fighter B, and A-minus-B differences.

**Plan**
1. Reshape: one row per *fighter per fight* (a "long" table), so we can look at each fighter's history in order.
2. For each row, compute stats from that fighter's *previous* fights only.
3. Join A's and B's features back onto each fight and compute differences.
4. Verify there's no leakage (brute-force check).

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

df = pd.read_csv("data/ufc_clean.csv", parse_dates=["event_date"])
print(df.shape, df.event_date.min().date(), "->", df.event_date.max().date())

(8780, 362) 1994-03-11 -> 2026-10-10


## 1. Fight duration
Per-minute stats need minutes fought. Duration = full rounds before the finish round (5 min each) + time in the final round.
*(Approximation: very early UFCs had different round lengths. Another reason to consider starting modeling around 2010.)*

In [2]:
t = df.finish_time.str.split(":", expand=True).astype(float)
df["dur_sec"] = (df.finish_round - 1) * 300 + t[0] * 60 + t[1]

print("Fights with missing duration:", df.dur_sec.isna().sum())
print(df.dur_sec.describe().round(0))

Fights with missing duration: 0


C:\Users\Sheld\AppData\Local\Temp\ipykernel_25664\1977007074.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["dur_sec"] = (df.finish_round - 1) * 300 + t[0] * 60 + t[1]


count    8780.0
mean      632.0
std       364.0
min         5.0
25%       275.0
50%       796.0
75%       900.0
max      1500.0
Name: dur_sec, dtype: float64


## 2. Reshape: one row per fighter per fight
For each fight we create two rows (fighter in slot 1, fighter in slot 2). Each row holds what the fighter did in that fight, plus what the *opponent* did to them (needed for "strikes absorbed" and takedown defense).

In [3]:
def side_rows(df, me, opp):
    r = range(1, 6)
    rd = lambda who, stat: df[[f"f_{who}_r{i}_{stat}" for i in r]].sum(axis=1)   # sums rounds, ignores missing rounds
    won = df.a_wins if me == 1 else 1 - df.a_wins
    return pd.DataFrame({
        "fight_url": df.fight_url,
        "event_date": df.event_date,
        "side": me,
        "fighter_id": df[f"f_{me}_url"],
        "won": won,
        "ko_win": ((won == 1) & (df.result == "KO/TKO")).astype(int),
        "sub_win": ((won == 1) & (df.result == "Submission")).astype(int),
        "dur_sec": df.dur_sec,
        # what I did
        "sig_land": df[f"f_{me}_sig_strikes_succ"], "sig_att": df[f"f_{me}_sig_strikes_att"],
        "td_land": df[f"f_{me}_takedown_succ"], "td_att": df[f"f_{me}_takedown_att"],
        "sub_att": df[f"f_{me}_submission_att"], "kd": df[f"f_{me}_knockdowns"],
        "ctrl_sec": df[f"f_{me}_ctrl_time_sec"],
        "dist_att": rd(me, "distance_att"), "clinch_att": rd(me, "clinch_att"), "ground_att": rd(me, "ground_att"),
        # what the opponent did to me
        "sig_absorbed": df[f"f_{opp}_sig_strikes_succ"],
        "opp_td_land": df[f"f_{opp}_takedown_succ"], "opp_td_att": df[f"f_{opp}_takedown_att"],
        # static attributes
        "height": df[f"f_{me}_fighter_height_cm"], "reach": df[f"f_{me}_fighter_reach_cm"],
        "stance": df[f"f_{me}_fighter_stance"],
        "dob": pd.to_datetime(df[f"f_{me}_fighter_dob"], errors="coerce"),
    })

long = pd.concat([side_rows(df, 1, 2), side_rows(df, 2, 1)], ignore_index=True)
long = long.sort_values(["fighter_id", "event_date", "fight_url"]).reset_index(drop=True)

print(long.shape)
# Early UFC events (1994-1999) were one-night tournaments: the same fighter can have 2-3 fights on one date,
# and the file has no fight order within a night. Let's see how much of that there is.
same_night = long[long.duplicated(["fighter_id", "event_date"], keep=False)]
print("Rows involving a fighter fighting twice on one date:", len(same_night))
print(same_night.groupby(same_night.event_date.dt.year).size().to_string())

# After 1999 it must never happen, otherwise our "previous fights" ordering would be ambiguous
assert not long[long.event_date >= "2000-01-01"].duplicated(["fighter_id", "event_date"]).any()
long.head(3)

(17560, 25)
Rows involving a fighter fighting twice on one date: 152
event_date
1994    34
1995    40
1996    38
1997    28
1998     8
1999     4


,fight_url,event_date,side,fighter_id,won,ko_win,sub_win,dur_sec,sig_land,sig_att,td_land,td_att,sub_att,kd,ctrl_sec,dist_att,clinch_att,ground_att,sig_absorbed,opp_td_land,opp_td_att,height,reach,stance,dob
0,http://ufcstats.com/fight-details/0e8599eca8cc...,2025-06-07,2,http://ufcstats.com/fighter-details/001eb2ab0f...,0,0,0,900.0,58,114,2,11,0,0,279,106.0,6.0,2.0,110,0,0,172.72,175.26,Orthodox,1995-10-22
1,http://ufcstats.com/fight-details/bc01ce748a57...,2019-07-20,1,http://ufcstats.com/fighter-details/002ca19647...,0,0,0,900.0,8,16,1,6,0,0,266,12.0,4.0,0.0,15,6,9,167.64,180.34,Orthodox,1994-08-26
2,http://ufcstats.com/fight-details/0e3af9b5e1b3...,2020-02-29,2,http://ufcstats.com/fighter-details/002ca19647...,0,0,0,900.0,31,97,2,6,1,0,148,89.0,3.0,5.0,89,2,4,167.64,180.34,Orthodox,1994-08-26


**What we do about it:** we still use 1994-1999 fights as *history* (they count as earlier fights for later careers),
but we will not use those tournament-era fights themselves as training rows, because we can't tell which came first on the night. See step 6.

## 3. Stats from *previous* fights only
Trick: a running total (`cumsum`) that **includes** the current fight, minus the current fight's own value, gives the total of everything *before* it.
Rates are then `total_before / minutes_before`.

In [4]:
sum_cols = ["won", "ko_win", "sub_win", "dur_sec", "sig_land", "sig_att", "sig_absorbed",
            "td_land", "td_att", "opp_td_land", "opp_td_att", "sub_att", "kd", "ctrl_sec",
            "dist_att", "clinch_att", "ground_att"]

# treat missing stat values as 0 so cumulative sums don't turn into NaN
long[sum_cols] = long[sum_cols].fillna(0)

g = long.groupby("fighter_id", sort=False)
for c in sum_cols:
    long[f"prev_{c}"] = g[c].cumsum() - long[c]       # total BEFORE this fight
long["n_prev"] = g.cumcount()                          # number of earlier UFC fights

def sdiv(a, b):
    return np.where(b > 0, a / b.where(b > 0, 1), np.nan)

mins = long.prev_dur_sec / 60
long["slpm"]         = sdiv(long.prev_sig_land, mins)                    # sig. strikes landed / min
long["sapm"]         = sdiv(long.prev_sig_absorbed, mins)                # sig. strikes absorbed / min
long["sig_acc"]      = sdiv(long.prev_sig_land, long.prev_sig_att)
long["td_per15"]     = sdiv(long.prev_td_land, long.prev_dur_sec / 900)
long["td_acc"]       = sdiv(long.prev_td_land, long.prev_td_att)
long["td_def"]       = 1 - sdiv(long.prev_opp_td_land, long.prev_opp_td_att)
long["sub_att_per15"]= sdiv(long.prev_sub_att, long.prev_dur_sec / 900)
long["kd_per15"]     = sdiv(long.prev_kd, long.prev_dur_sec / 900)
long["ctrl_share"]   = sdiv(long.prev_ctrl_sec, long.prev_dur_sec)
# style: how much of their striking happens in the clinch/on the ground vs at range
long["grappling_share"] = sdiv(long.prev_clinch_att + long.prev_ground_att,
                               long.prev_dist_att + long.prev_clinch_att + long.prev_ground_att)

long["prev_win_rate"] = sdiv(long.prev_won, long.n_prev)
long["ko_win_rate"]   = sdiv(long.prev_ko_win, long.n_prev)
long["sub_win_rate"]  = sdiv(long.prev_sub_win, long.n_prev)

## 4. Recent form, streaks, layoff, age

In [5]:
# win rate in last 3 fights (shift(1) = exclude the current fight)
long["last3_win_rate"] = g["won"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

# signed streak BEFORE the fight: +3 = on a 3-win streak, -2 = lost last 2
def streak_before(s):
    out, cur = [], 0
    for v in s:
        out.append(cur)
        cur = (cur + 1 if cur > 0 else 1) if v == 1 else (cur - 1 if cur < 0 else -1)
    return pd.Series(out, index=s.index)
long["streak"] = g["won"].transform(streak_before)

# layoff: days since previous UFC fight (NaN for a debut)
long["days_since_last"] = (long.event_date - g["event_date"].shift(1)).dt.days

# age at the time of the fight
long["age"] = (long.event_date - long.dob).dt.days / 365.25

long[["fighter_id", "event_date", "won", "n_prev", "prev_win_rate", "last3_win_rate",
      "streak", "days_since_last", "age"]].head(8)

,fighter_id,event_date,won,n_prev,prev_win_rate,last3_win_rate,streak,days_since_last,age
0,http://ufcstats.com/fighter-details/001eb2ab0f...,2025-06-07,0,0,NaN,NaN,0,NaN,29.626283
1,http://ufcstats.com/fighter-details/002ca19647...,2019-07-20,0,0,NaN,NaN,0,NaN,24.898015
2,http://ufcstats.com/fighter-details/002ca19647...,2020-02-29,0,1,0.0,0.0,-1,224.0,25.511294
3,http://ufcstats.com/fighter-details/003d82fa38...,2020-02-29,0,0,NaN,NaN,0,NaN,30.442163
4,http://ufcstats.com/fighter-details/003d82fa38...,2021-03-06,0,1,0.0,0.0,-1,371.0,31.457906
5,http://ufcstats.com/fighter-details/0052de9069...,2017-03-11,0,0,NaN,NaN,0,NaN,30.346338
6,http://ufcstats.com/fighter-details/0052de9069...,2017-12-09,1,1,0.0,0.0,-1,273.0,31.093771
7,http://ufcstats.com/fighter-details/0052de9069...,2018-05-12,1,2,0.5,0.5,1,154.0,31.515400


## 5. Brute-force leakage check
Recompute a few features the slow, obvious way (loop over each fighter's earlier fights) for 300 random rows and confirm they match. If this passes, the cumsum trick is doing what we think.

In [6]:
rng = np.random.default_rng(0)
sample = long[long.n_prev >= 1].sample(300, random_state=0)

for idx, row in sample.iterrows():
    earlier = long[(long.fighter_id == row.fighter_id) & (long.event_date < row.event_date)]
    assert len(earlier) == row.n_prev
    assert abs(earlier.won.mean() - row.prev_win_rate) < 1e-9
    assert abs(earlier.sig_land.sum() / (earlier.dur_sec.sum() / 60) - row.slpm) < 1e-9
    assert earlier.won.tail(3).mean() == row.last3_win_rate
    assert (row.event_date - earlier.event_date.max()).days == row.days_since_last

debuts = long[long.n_prev == 0]
assert debuts[["slpm", "prev_win_rate", "days_since_last"]].isna().all().all()
print("Leakage checks passed on 300 random rows + all", len(debuts), "debut rows")

Leakage checks passed on 300 random rows + all 2760 debut rows


## 6. Join A and B back onto each fight and build differences

In [7]:
num_feats = ["n_prev", "prev_win_rate", "last3_win_rate", "streak", "days_since_last", "age",
             "height", "reach", "slpm", "sapm", "sig_acc", "td_per15", "td_acc", "td_def",
             "sub_att_per15", "kd_per15", "ctrl_share", "grappling_share", "ko_win_rate", "sub_win_rate"]
cat_feats = ["stance"]

A = long[long.side == 1].set_index("fight_url")[num_feats + cat_feats].add_prefix("a_")
B = long[long.side == 2].set_index("fight_url")[num_feats + cat_feats].add_prefix("b_")

fights = (df.set_index("fight_url")[["event_date", "weight_class", "title_fight", "num_rounds",
                                      "f_1_name", "f_2_name", "a_wins"]]
            .join(A).join(B))

for c in num_feats:
    fights[f"diff_{c}"] = fights[f"a_{c}"] - fights[f"b_{c}"]      # positive = A has more of it

# tournament era (1994-1999): within-night fight order is unknown, so don't use these as modeling rows
print("Fights dropped as tournament-era (before 2000):", (fights.event_date < "2000-01-01").sum())
fights = fights[fights.event_date >= "2000-01-01"].copy()

fights["same_stance"] = (fights.a_stance == fights.b_stance).astype(int)
fights["a_reach_missing"] = fights.a_reach.isna().astype(int)
fights["b_reach_missing"] = fights.b_reach.isna().astype(int)
fights["min_prev_fights"] = fights[["a_n_prev", "b_n_prev"]].min(axis=1)

print(fights.shape)
fights.head(3).T.head(30)

Fights dropped as tournament-era (before 2000): 219
(8561, 73)


fight_url,http://ufcstats.com/fight-details/9f2b94c46b70c6fb,http://ufcstats.com/fight-details/aca2f98ee112d137,http://ufcstats.com/fight-details/22911cbd32370f6c
event_date,2000-03-10 00:00:00,2000-03-10 00:00:00,2000-03-10 00:00:00
weight_class,Heavyweight,Lightweight,Lightweight
title_fight,False,False,False
num_rounds,3.0,3.0,3.0
f_1_name,Tedd Williams,Dave Menne,Bob Cook
f_2_name,Steve Judson,Fabiano Iha,Tiki Ghosn
a_wins,1,1,1
a_n_prev,0,0,0
a_prev_win_rate,NaN,NaN,NaN
a_last3_win_rate,NaN,NaN,NaN


## 7. Cold start: how many fights have enough history?
Fighters on their first few UFC fights have little or no history, so their features are empty or noisy. Decide a cutoff for the model.

In [8]:
for k in [0, 1, 2, 3, 5]:
    n = (fights.min_prev_fights >= k).sum()
    print(f"both fighters have >= {k} prior UFC fights: {n:5d} fights ({n/len(fights):.0%})")

# And how much data per year survives a >=3 cutoff
ok = fights[fights.min_prev_fights >= 3]
print()
print(ok.groupby(ok.event_date.dt.year).size().tail(12).to_string())

both fighters have >= 0 prior UFC fights:  8561 fights (100%)
both fighters have >= 1 prior UFC fights:  6468 fights (76%)
both fighters have >= 2 prior UFC fights:  4937 fights (58%)
both fighters have >= 3 prior UFC fights:  3881 fights (45%)
both fighters have >= 5 prior UFC fights:  2488 fights (29%)

event_date
2015    200
2016    244
2017    214
2018    230
2019    225
2020    210
2021    257
2022    286
2023    263
2024    304
2025    284
2026    232


## 8. Do these features carry any signal? (descriptive check, not a model)
Among fights where both fighters have 3+ prior fights: how often does fighter A win when A has the higher value of each feature? 50% = no information. This is just a sanity check that the features aren't empty noise.

In [9]:
rows = []
for c in ["prev_win_rate", "last3_win_rate", "streak", "age", "reach", "slpm", "sapm",
          "sig_acc", "td_per15", "td_def", "ctrl_share", "days_since_last"]:
    d = ok[ok[f"diff_{c}"].notna() & (ok[f"diff_{c}"] != 0)]
    rows.append({"feature": c, "n_fights": len(d), "A_wins_when_A_higher": round((d.a_wins[d[f"diff_{c}"] > 0]).mean(), 3)})
pd.DataFrame(rows).sort_values("A_wins_when_A_higher", ascending=False)

,feature,n_fights,A_wins_when_A_higher
0,prev_win_rate,3610,0.575
1,last3_win_rate,2312,0.569
7,sig_acc,3881,0.545
2,streak,2964,0.544
8,td_per15,3850,0.543
9,td_def,3730,0.533
10,ctrl_share,3881,0.530
5,slpm,3881,0.526
4,reach,3339,0.522
11,days_since_last,3676,0.491


## 9. Save

In [10]:
fights.reset_index().to_csv("data/ufc_features.csv", index=False)
print("Saved", fights.shape, "-> data/ufc_features.csv")

Saved (8561, 73) -> data/ufc_features.csv


**Next (notebook 3):** train/test split **by date** (train on older fights, test on newer ones — never random, or you leak the future), then a logistic regression on the `diff_*` columns, and an honest accuracy number.